In [3]:
import pandas as pd
import numpy as np
import random
import time

# ==== Load data (with optional VPE cols e, f) ====
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/3-unit.xlsx',
    sheet_name='Table 2'
)
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# VPE parameters (optional)
if 'd' in data.columns and 'e' in data.columns:
    d = data['d'].values.astype(float)
    e = data['e'].values.astype(float)
    _vpe_warned = False
else:
    # If missing, fall back to zeros (VPE off)
    d = np.zeros_like(a, dtype=float)
    e = np.zeros_like(a, dtype=float)
    _vpe_warned = True
    print("[INFO] Kolom d/e tidak ditemukan. VPE dinonaktifkan (diasumsikan d=e=0).")

# Load power demand
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd3u.xlsx')
power_demand = power_demand_data['load'].values

# ==== Cost functions with VPE ====
def vpe_term(P, pmin, d, e):
    """
    | e_i * sin( f_i * (pmin_i - P_i) ) |
    """
    return np.abs(d * np.sin(e * (pmin - P)))

def unit_cost(P, a, b, c, pmin, d, e):
    """Per-unit cost including VPE (vector)."""
    return a + b*P + c*(P**2) + vpe_term(P, pmin, d, e)

def total_cost(P, a, b, c, pmin, d, e):
    """Scalar total cost including VPE."""
    return np.sum(unit_cost(P, a, b, c, pmin, d, e))

# ==== DOA-based power calculation (objective updated with VPE) ====
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100, penalty_mismatch=10000.0):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost (with VPE) + penalty mismatch
    def fobj(pos):
        fuel = total_cost(pos, a, b, c, lb, d, e)
        penalty = abs(demand - np.sum(pos)) * penalty_mismatch
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# Fuel cost calculation (now includes VPE)
def calculate_fuel_cost(P, a, b, c):
    """
    Return per-unit and total fuel cost including VPE.
    Kept signature the same for compatibility with the rest of the code.
    """
    per_unit = unit_cost(P, a, b, c, Minimum_Capacity, d, e)
    total = float(np.sum(per_unit))
    return per_unit, total

# ==== Main loop ====
all_outputs = []
schedulling = []
summary_rows = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost (with VPE): Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp) with VPE': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost (with VPE)'] = Fuel_Cost[unit_index]
    schedulling.append(output_info)

# Ramp rates & generation limits
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ==== Save to Excel ====
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED-VPE) 3u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED-VPE) 3u.xlsx' (biaya termasuk VPE)")

Load Counter: 1
Output for Power Demand 850.00 MW
   Unit  Power Produced (MW)
0     1           433.398384
1     2           248.974628
2     3           167.626988
Total Power Produced: 850.00 MW
Total Fuel Cost (with VPE): Rp 8625.98334171592
Computation Time: 0.038574934005737305 seconds
------------------------------------------------

All results saved to 'DOA (ED-VPE) 3u.xlsx' (biaya termasuk VPE)


In [4]:
import pandas as pd
import numpy as np
import random
import time

# ==== Load data (with optional VPE cols e, f) ====
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/15-unit.xlsx',
    sheet_name='Table 4'
)
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# VPE parameters (optional)
if 'd' in data.columns and 'e' in data.columns:
    d = data['d'].values.astype(float)
    e = data['e'].values.astype(float)
    _vpe_warned = False
else:
    # If missing, fall back to zeros (VPE off)
    d = np.zeros_like(a, dtype=float)
    e = np.zeros_like(a, dtype=float)
    _vpe_warned = True
    print("[INFO] Kolom d/e tidak ditemukan. VPE dinonaktifkan (diasumsikan d=e=0).")

# Load power demand
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd15u.xlsx')
power_demand = power_demand_data['load'].values

# ==== Cost functions with VPE ====
def vpe_term(P, pmin, d, e):
    """
    | e_i * sin( f_i * (pmin_i - P_i) ) |
    """
    return np.abs(d * np.sin(e * (pmin - P)))

def unit_cost(P, a, b, c, pmin, d, e):
    """Per-unit cost including VPE (vector)."""
    return a + b*P + c*(P**2) + vpe_term(P, pmin, d, e)

def total_cost(P, a, b, c, pmin, d, e):
    """Scalar total cost including VPE."""
    return np.sum(unit_cost(P, a, b, c, pmin, d, e))

# ==== DOA-based power calculation (objective updated with VPE) ====
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100, penalty_mismatch=10000.0):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost (with VPE) + penalty mismatch
    def fobj(pos):
        fuel = total_cost(pos, a, b, c, lb, d, e)
        penalty = abs(demand - np.sum(pos)) * penalty_mismatch
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# Fuel cost calculation (now includes VPE)
def calculate_fuel_cost(P, a, b, c):
    """
    Return per-unit and total fuel cost including VPE.
    Kept signature the same for compatibility with the rest of the code.
    """
    per_unit = unit_cost(P, a, b, c, Minimum_Capacity, d, e)
    total = float(np.sum(per_unit))
    return per_unit, total

# ==== Main loop ====
all_outputs = []
schedulling = []
summary_rows = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost (with VPE): Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp) with VPE': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost (with VPE)'] = Fuel_Cost[unit_index]
    schedulling.append(output_info)

# Ramp rates & generation limits
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ==== Save to Excel ====
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED-VPE) 15u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED-VPE) 15u.xlsx' (biaya termasuk VPE)")

Load Counter: 1
Output for Power Demand 2630.00 MW
    Unit  Power Produced (MW)
0      1           357.774471
1      2           374.930371
2      3           101.536371
3      4            82.112065
4      5           371.353609
5      6           275.033249
6      7           453.116191
7      8           224.277853
8      9            93.718238
9     10            91.580190
10    11            46.079512
11    12            29.458830
12    13            66.588116
13    14            35.646561
14    15            26.794371
Total Power Produced: 2630.00 MW
Total Fuel Cost (with VPE): Rp 34514.56018121561
Computation Time: 0.04042220115661621 seconds
------------------------------------------------

All results saved to 'DOA (ED-VPE) 15u.xlsx' (biaya termasuk VPE)


In [5]:
import pandas as pd
import numpy as np
import random
import time

# ==== Load data (with optional VPE cols e, f) ====
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/40-unit.xlsx',
    sheet_name='Table 2'
)
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# VPE parameters (optional)
if 'd' in data.columns and 'e' in data.columns:
    d = data['d'].values.astype(float)
    e = data['e'].values.astype(float)
    _vpe_warned = False
else:
    # If missing, fall back to zeros (VPE off)
    d = np.zeros_like(a, dtype=float)
    e = np.zeros_like(a, dtype=float)
    _vpe_warned = True
    print("[INFO] Kolom d/e tidak ditemukan. VPE dinonaktifkan (diasumsikan d=e=0).")

# Load power demand
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd40u.xlsx')
power_demand = power_demand_data['load'].values

# ==== Cost functions with VPE ====
def vpe_term(P, pmin, d, e):
    """
    | e_i * sin( f_i * (pmin_i - P_i) ) |
    """
    return np.abs(d * np.sin(e * (pmin - P)))

def unit_cost(P, a, b, c, pmin, d, e):
    """Per-unit cost including VPE (vector)."""
    return a + b*P + c*(P**2) + vpe_term(P, pmin, d, e)

def total_cost(P, a, b, c, pmin, d, e):
    """Scalar total cost including VPE."""
    return np.sum(unit_cost(P, a, b, c, pmin, d, e))

# ==== DOA-based power calculation (objective updated with VPE) ====
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100, penalty_mismatch=10000.0):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost (with VPE) + penalty mismatch
    def fobj(pos):
        fuel = total_cost(pos, a, b, c, lb, d, e)
        penalty = abs(demand - np.sum(pos)) * penalty_mismatch
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# Fuel cost calculation (now includes VPE)
def calculate_fuel_cost(P, a, b, c):
    """
    Return per-unit and total fuel cost including VPE.
    Kept signature the same for compatibility with the rest of the code.
    """
    per_unit = unit_cost(P, a, b, c, Minimum_Capacity, d, e)
    total = float(np.sum(per_unit))
    return per_unit, total

# ==== Main loop ====
all_outputs = []
schedulling = []
summary_rows = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost (with VPE): Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp) with VPE': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost (with VPE)'] = Fuel_Cost[unit_index]
    schedulling.append(output_info)

# Ramp rates & generation limits
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ==== Save to Excel ====
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED-VPE) 40u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED-VPE) 40u.xlsx' (biaya termasuk VPE)")

Load Counter: 1
Output for Power Demand 10500.00 MW
    Unit  Power Produced (MW)
0      1            56.417481
1      2            88.297084
2      3            99.464102
3      4           127.456291
4      5            72.902989
5      6            74.165231
6      7           269.872990
7      8           282.169669
8      9           221.836676
9     10           216.793921
10    11           334.182050
11    12           274.347851
12    13           461.245700
13    14           425.933275
14    15           491.584343
15    16           405.291529
16    17           470.043269
17    18           329.280576
18    19           543.062994
19    20           530.259876
20    21           492.937280
21    22           443.284717
22    23           465.341491
23    24           286.770918
24    25           399.379984
25    26           531.918303
26    27           129.079172
27    28           128.893032
28    29            53.928918
29    30            59.482022
30    31          